In [1]:
# Data preprocessing in python for machine learning

In [2]:
import re
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
import warnings; warnings.filterwarnings('ignore')
from sklearn.datasets import load_wine
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

In [3]:
planes_df = pd.read_csv('Data/planes_crashes.csv')
planes_df.head()

,Date,Time,Location,Operator,Flight #,Route,Type,Registration,cn/In,Aboard,Fatalities,Ground,Summary
0,09/17/1908,17:18,"Fort Myer, Virginia",Military - U.S. Army,NaN,Demonstration,Wright Flyer III,NaN,1,2.0,1.0,0.0,"During a demonstration flight, a U.S. Army fly..."
1,07/12/1912,06:30,"AtlantiCity, New Jersey",Military - U.S. Navy,NaN,Test flight,Dirigible,NaN,NaN,5.0,5.0,0.0,First U.S. dirigible Akron exploded just offsh...
2,08/06/1913,NaN,"Victoria, British Columbia, Canada",Private,-,NaN,Curtiss seaplane,NaN,NaN,1.0,1.0,0.0,The first fatal airplane accident in Canada oc...
3,09/09/1913,18:30,Over the North Sea,Military - German Navy,NaN,NaN,Zeppelin L-1 (airship),NaN,NaN,20.0,14.0,0.0,The airship flew into a thunderstorm and encou...
4,10/17/1913,10:30,"Near Johannisthal, Germany",Military - German Navy,NaN,NaN,Zeppelin L-2 (airship),NaN,NaN,30.0,30.0,0.0,Hydrogen gas which was being vented was sucked...


In [4]:
print(planes_df.isna().sum())

Date               0
Time            2219
Location          20
Operator          18
Flight #        4199
Route           1707
Type              27
Registration     335
cn/In           1228
Aboard            22
Fatalities        12
Ground            22
Summary          390
dtype: int64


In [5]:
print(planes_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5268 entries, 0 to 5267
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          5268 non-null   object 
 1   Time          3049 non-null   object 
 2   Location      5248 non-null   object 
 3   Operator      5250 non-null   object 
 4   Flight #      1069 non-null   object 
 5   Route         3561 non-null   object 
 6   Type          5241 non-null   object 
 7   Registration  4933 non-null   object 
 8   cn/In         4040 non-null   object 
 9   Aboard        5246 non-null   float64
 10  Fatalities    5256 non-null   float64
 11  Ground        5246 non-null   float64
 12  Summary       4878 non-null   object 
dtypes: float64(3), object(10)
memory usage: 535.2+ KB
None


In [6]:
planes_df["Date"] = pd.to_datetime(planes_df["Date"])

In [7]:
# Drop the cn/In and Route columns from planes_df (less useful for most analysis)
planes_cols = planes_df.drop(["cn/In", "Route"], axis=1)

# Drop rows with missing Fatalities values from planes_cols
planes_subset = planes_cols.dropna(subset=["Fatalities"])

# Print out the shape of the subset
print(planes_subset.shape)

(5256, 11)


In [8]:
print(planes_subset.isna().sum())

Date               0
Time            2207
Location          19
Operator          18
Flight #        4187
Type              25
Registration     334
Aboard            10
Fatalities         0
Ground            10
Summary          385
dtype: int64


In [9]:
planes_df['Operator'].dropna()

0                  Military - U.S. Army
1                  Military - U.S. Navy
2                               Private
3                Military - German Navy
4                Military - German Navy
                     ...               
5263    Military - Indonesian Air Force
5264                        Service Air
5265                         Air France
5266                         Strait Air
5267        Military - Indian Air Force
Name: Operator, Length: 5250, dtype: object

In [10]:
# Drop rows with missing Operator (can't train/predict a missing label)
planes_df = planes_df.dropna(subset=['Operator'])

# Keep only operators with at least 2 occurrences (required for stratified split)
operator_counts = planes_df['Operator'].value_counts()
valid_operators = operator_counts[operator_counts >= 2].index
planes_df = planes_df[planes_df['Operator'].isin(valid_operators)]

# Create a DataFrame with all columns except Operator
X = planes_df.drop("Operator", axis=1)

# Create an Operator labels dataset
y = planes_df[['Operator']]

# Use stratified sampling to split up the dataset according to the y dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Print the Operator counts from y_train
print(y_train['Operator'].value_counts())

Operator
Aeroflot                             134
Military - U.S. Air Force            132
Air France                            52
Deutsche Lufthansa                    49
Air Taxi                              33
                                    ... 
Saturn Airways                         1
Salair                                 1
Crossair                               1
Channel Flying Service - Air Taxi      1
Military - Colombian Army              1
Name: count, Length: 555, dtype: int64


In [11]:
# Sample data
x = np.array([1, 2, 3, 4, 5])
y = np.array([2, 4, 5, 4, 5])

# Add constant for intercept term
X = sm.add_constant(x)

# Fit OLS model
model = sm.OLS(y, X).fit()

# Print summary
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.600
Model:                            OLS   Adj. R-squared:                  0.467
Method:                 Least Squares   F-statistic:                     4.500
Date:                Thu, 30 Jul 2026   Prob (F-statistic):              0.124
Time:                        06:48:22   Log-Likelihood:                -5.2598
No. Observations:                   5   AIC:                             14.52
Df Residuals:                       3   BIC:                             13.74
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          2.2000      0.938      2.345      0.1

In [12]:
# Load the wine dataset
wine = load_wine()
X = wine.data
y = wine.target

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

knn = KNeighborsClassifier()

# Fit the knn model to the training data
knn.fit(X_train, y_train)

# Score the model on the test data
print(knn.score(X_test, y_test))

0.7777777777777778


We see that without normalization, our accuracy score for the model is very low so we have to standardize it.

In [13]:
wine_df = pd.DataFrame(wine.data, columns=wine.feature_names)
print(wine_df.columns.tolist())

['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']


In [14]:
print(wine_df.var())

alcohol                             0.659062
malic_acid                          1.248015
ash                                 0.075265
alcalinity_of_ash                  11.152686
magnesium                         203.989335
total_phenols                       0.391690
flavanoids                          0.997719
nonflavanoid_phenols                0.015489
proanthocyanins                     0.327595
color_intensity                     5.374449
hue                                 0.052245
od280/od315_of_diluted_wines        0.504086
proline                         99166.717355
dtype: float64


In [15]:
# Print out the variance of the Proline column
print(wine_df['proline'].var())

# Apply the log normalization function to the Proline column
wine_df['Proline_log'] = np.log(wine_df['proline'])

# Check the variance of the normalized Proline column
print(wine_df['Proline_log'].var())

99166.71735542428
0.17231366191842018


In [16]:
# Create the scaler
scaler = StandardScaler()

# Subset the DataFrame you want to scale 
wine_subset = wine_df[['ash', 'alcalinity_of_ash', 'magnesium']]

# Apply the scaler to wine_subset
wine_subset_scaled = scaler.fit_transform(wine_subset)

In [17]:
# Scale the training and test features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit the k-nearest neighbors model to the training data
knn.fit(X_train_scaled, y_train)

# Score the model on the test data
print(knn.score(X_test_scaled, y_test))

0.9333333333333333


In [18]:
churn_df = pd.read_csv('Data/telecom_churn.csv')
churn_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [19]:
print(churn_df['PhoneService'].nunique()) # This feature needs to be encoded 

2


In [20]:
# Set up the LabelEncoder object
enc = LabelEncoder()

# Apply the encoding to the "PhoneService" column
churn_df["PhoneService_enc"] = enc.fit_transform(churn_df["PhoneService"])

# Compare the two columns
print(churn_df[["PhoneService", "PhoneService_enc"]].head())

  PhoneService  PhoneService_enc
0           No                 0
1          Yes                 1
2          Yes                 1
3           No                 0
4          Yes                 1


In [21]:
print(churn_df['Churn'].unique()) # This label needs encoding

['No' 'Yes']


In [22]:
demo_df = pd.get_dummies(churn_df['Churn'], dtype=int)
demo_df.head(5)

,No,Yes
0,1,0
1,1,0
2,0,1
3,1,0
4,0,1


In [23]:
# Apply the encoding to the "Churn" column
churn_df["Churn_enc"] = enc.fit_transform(churn_df["Churn"])

# Compare the two columns
print(churn_df[["Churn", "Churn_enc"]].head())

  Churn  Churn_enc
0    No          0
1    No          0
2   Yes          1
3    No          0
4   Yes          1


In [24]:
# Transform the category_desc column
category_enc = pd.get_dummies(churn_df["PaperlessBilling"])

# Take a look at the encoded columns
print(category_enc.head())

      No    Yes
0  False   True
1   True  False
2  False   True
3   True  False
4  False   True


In [25]:
wine_df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,Proline_log
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,6.970730
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,6.956545
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,7.077498
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,7.299797
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,6.599870


In [26]:
churn_df["PaperlessBilling_enc"] = churn_df["PaperlessBilling"].apply(lambda val: 1 if val == "Yes" else 0)
churn_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,PhoneService_enc,Churn_enc,PaperlessBilling_enc
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0,0,1
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,One year,No,Mailed check,56.95,1889.5,No,1,0,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,1,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0,0,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,1,1


In [27]:
np.random.seed(42)
n = 200

Run1 = np.random.uniform(0, 300, n)
Run2 = np.random.uniform(0, 250, n)
Run3 = np.random.uniform(0, 100, n)
Run4 = np.random.uniform(0, 150, n)

df = pd.DataFrame({'Run1': Run1, 'Run2': Run2, 'Run3': Run3, 'Run4': Run4})
print(df.describe().round(2))

         Run1    Run2    Run3    Run4
count  200.00  200.00  200.00  200.00
mean   145.20  126.09   52.07   71.82
std     88.47   73.25   30.73   41.83
min      1.66    1.27    1.08    2.72
25%     68.57   65.37   25.55   35.40
50%    148.35  135.41   52.54   70.07
75%    227.06  185.55   81.11  108.66
max    296.07  247.63   99.97  148.49


In [28]:
# Use .loc to create a mean column
df["mean"] = df.loc[:, "Run1":"Run4"].mean(axis=1)

# Take a look at the results
print(df.head())

         Run1        Run2       Run3       Run4        mean
0  112.362036  160.507912  10.312387  25.340259   77.130648
1  285.214292   21.034991  90.255291  41.788551  109.573281
2  219.598183   40.407179  50.525237  26.551573   84.270543
3  179.597545  224.638547  82.645747  13.305380  125.046805
4   46.805592  151.607265  32.004960  18.095381   62.128299


In [29]:
df_miles = df.copy() ## Copying the dataframe, converting the datatype to string, and appending mile at the end of the observations

for col in ['Run1', 'Run2', 'Run3', 'Run4']:
    df_miles[col] = df_miles[col].round(3).astype(str) + " mile"

print(df_miles.head())

           Run1          Run2         Run3         Run4        mean
0  112.362 mile  160.508 mile  10.312 mile   25.34 mile   77.130648
1  285.214 mile   21.035 mile  90.255 mile  41.789 mile  109.573281
2  219.598 mile   40.407 mile  50.525 mile  26.552 mile   84.270543
3  179.598 mile  224.639 mile  82.646 mile  13.305 mile  125.046805
4   46.806 mile  151.607 mile  32.005 mile  18.095 mile   62.128299


In [30]:
# First, convert string column to date column
planes_df["Date_converted"] = pd.to_datetime(planes_df['Date'])

# Extract just the month from the converted column
planes_df["date_month"] = planes_df['Date_converted'].dt.month

# Extract just the month from the converted column
planes_df["date_day"] = planes_df['Date_converted'].dt.day

# Take a look at the converted and new month columns
print(planes_df[['Date_converted', 'date_month', 'date_day']].head())

  Date_converted  date_month  date_day
0     1908-09-17           9        17
1     1912-07-12           7        12
2     1913-08-06           8         6
3     1913-09-09           9         9
4     1913-10-17          10        17


In [31]:
# Write a pattern to extract numbers and decimals
def return_mileage(length):
    
    # Search the text for matches
    mile = re.search(r"\d+\.?\d*", length)
    
    # If a value is returned, use group(0) to return the found value
    if mile is not None:
        return float(mile.group(0))
        
# Apply the function to the Run1 column and take a look at both columns
df_miles["Run1_num"] = df_miles["Run1"].apply(return_mileage)
print(df_miles[["Run1", "Run1_num"]].head())

           Run1  Run1_num
0  112.362 mile   112.362
1  285.214 mile   285.214
2  219.598 mile   219.598
3  179.598 mile   179.598
4   46.806 mile    46.806


In [32]:
planes_df['severity'] = pd.cut(planes_df['Fatalities'], bins=[0, 10, 50, np.inf], labels=['low', 'moderate', 'severe'])
planes_df.head()

,Date,Time,Location,Operator,Flight #,Route,Type,Registration,cn/In,Aboard,Fatalities,Ground,Summary,Date_converted,date_month,date_day,severity
0,1908-09-17,17:18,"Fort Myer, Virginia",Military - U.S. Army,NaN,Demonstration,Wright Flyer III,NaN,1,2.0,1.0,0.0,"During a demonstration flight, a U.S. Army fly...",1908-09-17,9,17,low
1,1912-07-12,06:30,"AtlantiCity, New Jersey",Military - U.S. Navy,NaN,Test flight,Dirigible,NaN,NaN,5.0,5.0,0.0,First U.S. dirigible Akron exploded just offsh...,1912-07-12,7,12,low
2,1913-08-06,NaN,"Victoria, British Columbia, Canada",Private,-,NaN,Curtiss seaplane,NaN,NaN,1.0,1.0,0.0,The first fatal airplane accident in Canada oc...,1913-08-06,8,6,low
3,1913-09-09,18:30,Over the North Sea,Military - German Navy,NaN,NaN,Zeppelin L-1 (airship),NaN,NaN,20.0,14.0,0.0,The airship flew into a thunderstorm and encou...,1913-09-09,9,9,moderate
4,1913-10-17,10:30,"Near Johannisthal, Germany",Military - German Navy,NaN,NaN,Zeppelin L-2 (airship),NaN,NaN,30.0,30.0,0.0,Hydrogen gas which was being vented was sucked...,1913-10-17,10,17,moderate


In [33]:
# Drop rows missing Summary or severity
planes_df_clean = planes_df.dropna(subset=['Summary', 'severity'])

# Take the Summary text
title_text = planes_df_clean["Summary"]

# Create the vectorizer method
tfidf_vec = TfidfVectorizer()

# Transform the text into tf-idf vectors
text_tfidf = tfidf_vec.fit_transform(title_text)

# Split the dataset according to the class distribution of severity
y = planes_df_clean["severity"]
X_train, X_test, y_train, y_test = train_test_split(text_tfidf.toarray(), y, stratify=y, random_state=42)

# Fit the model
nb = MultinomialNB()
nb.fit(X_train, y_train)

# Print out the model's accuracy
print(nb.score(X_test, y_test))

0.606951871657754


In [34]:
print(y_test.value_counts(normalize=True))

severity
low         0.439840
moderate    0.437166
severe      0.122995
Name: proportion, dtype: float64


The Naive Bayes model achieved 60.7% accuracy predicting crash severity from summary text alone — a meaningful ~17-point lift over the 44.0% baseline of guessing the majority class (low), confirming the text carries real predictive signal. However, with low and moderate nearly tied in frequency and likely using overlapping language, and severe underrepresented at just 12.3%, the model's imperfect performance suggests combining text with structured features (e.g. Aboard, Ground) could improve results further.

In [35]:
# Create a list of redundant column names to drop
to_drop = ["PhoneService", "Churn", "PaperlessBilling"]

# Drop those columns from the dataset
churn_subset = churn_df.drop(to_drop, axis=1)

# Print out the head of volunteer_subset
churn_subset.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaymentMethod,MonthlyCharges,TotalCharges,PhoneService_enc,Churn_enc,PaperlessBilling_enc
0,7590-VHVEG,Female,0,Yes,No,1,No phone service,DSL,No,Yes,...,No,No,No,Month-to-month,Electronic check,29.85,29.85,0,0,1
1,5575-GNVDE,Male,0,No,No,34,No,DSL,Yes,No,...,No,No,No,One year,Mailed check,56.95,1889.5,1,0,0
2,3668-QPYBK,Male,0,No,No,2,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Mailed check,53.85,108.15,1,1,1
3,7795-CFOCW,Male,0,No,No,45,No phone service,DSL,Yes,No,...,Yes,No,No,One year,Bank transfer (automatic),42.30,1840.75,0,0,0
4,9237-HQITU,Female,0,No,No,2,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Electronic check,70.70,151.65,1,1,1


In [36]:
# Print out the column correlations of the wine dataset
print(wine_df.corr())

# Drop that column from the DataFrame
wine = wine_df.drop("flavanoids", axis=1)

print(wine_df.head())

                               alcohol  malic_acid       ash  \
alcohol                       1.000000    0.094397  0.211545   
malic_acid                    0.094397    1.000000  0.164045   
ash                           0.211545    0.164045  1.000000   
alcalinity_of_ash            -0.310235    0.288500  0.443367   
magnesium                     0.270798   -0.054575  0.286587   
total_phenols                 0.289101   -0.335167  0.128980   
flavanoids                    0.236815   -0.411007  0.115077   
nonflavanoid_phenols         -0.155929    0.292977  0.186230   
proanthocyanins               0.136698   -0.220746  0.009652   
color_intensity               0.546364    0.248985  0.258887   
hue                          -0.071747   -0.561296 -0.074667   
od280/od315_of_diluted_wines  0.072343   -0.368710  0.003911   
proline                       0.643720   -0.192011  0.223626   
Proline_log                   0.637325   -0.152643  0.238394   

                              alcalinit

In [37]:
vocab = { 1048: 'web',
 278: 'designer',
 1017: 'urban',
 38: 'adventures',
 490: 'ice',
 890: 'skating',
 90: 'at',
 559: 'lasker',
 832: 'rink',
 368: 'fight',
 423: 'global',
 487: 'hunger',
 68: 'and',
 944: 'support',
 1061: 'women',
 356: 'farmers',
 535: 'join',
 969: 'the',
 708: 'oxfam',
 27: 'action',
 240: 'corps',
 498: 'in',
 680: 'nyc',
 922: 'stop',
 947: 'swap',
 790: 'queens',
 911: 'staff',
 281: 'development',
 992: 'trainer',
 200: 'claro',
 145: 'brooklyn',
 1037: 'volunteer',
 93: 'attorney',
 221: 'community',
 455: 'health',
 43: 'advocates',
 942: 'supervise',
 189: 'children',
 466: 'highland',
 717: 'park',
 409: 'garden',
 1071: 'worldofmoney',
 696: 'org',
 1085: 'youth',
 60: 'amazing',
 791: 'race',
 789: 'qualified',
 133: 'board',
 620: 'member',
 860: 'seats',
 98: 'available',
 1083: 'young',
 33: 'adult',
 1006: 'tutor',
 1016: 'updated',
 11: '30',
 0: '11',
 513: 'insurance',
 199: 'claims',
 600: 'manager',
 979: 'timebanksnyc',
 432: 'great',
 340: 'exchange',
 205: 'clean',
 1015: 'up',
 81: 'asbury',
 171: 'cementary',
 918: 'staten',
 524: 'island',
 869: 'senior',
 194: 'citizen',
 392: 'friendly',
 1033: 'visitor',
 881: 'shop',
 1000: 'tree',
 161: 'care',
 1068: 'workshop',
 4: '20',
 646: 'movie',
 856: 'screener',
 380: 'for',
 870: 'seniors',
 355: 'farm',
 430: 'graphic',
 691: 'open',
 480: 'house',
 416: 'get',
 984: 'tools',
 980: 'to',
 806: 'recycling',
 1039: 'volunteers',
 660: 'needed',
 353: 'family',
 336: 'event',
 207: 'clerical',
 158: 'cancer',
 1041: 'walk',
 120: 'befitnyc',
 739: 'physical',
 30: 'activity',
 700: 'organizers',
 269: 'decision',
 266: 'day',
 5: '2011',
 661: 'needs',
 1084: 'your',
 459: 'help',
 405: 'gain',
 1021: 'valuable',
 245: 'counseling',
 344: 'experience',
 687: 'on',
 845: 'samaritans',
 9: '24',
 479: 'hour',
 255: 'crisis',
 478: 'hotline',
 457: 'heart',
 407: 'gallery',
 703: 'our',
 503: 'info',
 949: 'table',
 373: 'finding',
 471: 'homes',
 542: 'kids',
 1077: 'yiddish',
 903: 'speaking',
 472: 'homework',
 460: 'helper',
 892: 'skilled',
 800: 'rebuilding',
 982: 'together',
 468: 'home',
 818: 'repairs',
 438: 'greenteam',
 40: 'advetures',
 940: 'summer',
 931: 'streets',
 1005: 'tuesday',
 335: 'evenings',
 1060: 'with',
 612: 'masa',
 594: 'lunch',
 770: 'program',
 1018: 'us',
 706: 'outreach',
 618: 'meals',
 760: 'preparedness',
 222: 'compost',
 773: 'project',
 613: 'master',
 223: 'composter',
 178: 'certificate',
 249: 'course',
 318: 'emblemhealth',
 144: 'bronx',
 683: 'of',
 873: 'service',
 531: 'jcc',
 601: 'manhattan',
 418: 'girl',
 855: 'scout',
 872: 'series',
 296: 'dorot',
 838: 'rosh',
 452: 'hashanah',
 709: 'package',
 274: 'delivery',
 713: 'painting',
 511: 'instructor',
 530: 'jasa',
 464: 'hes',
 172: 'center',
 12: '3rd',
 70: 'annual',
 377: 'flyny',
 548: 'kite',
 366: 'festival',
 983: 'tomorrow',
 151: 'business',
 566: 'leaders',
 955: 'teach',
 110: 'basics',
 465: 'high',
 852: 'schoolers',
 410: 'gardening',
 397: 'ft',
 1004: 'tryon',
 910: 'st',
 610: 'martin',
 748: 'poetry',
 668: 'new',
 1079: 'york',
 216: 'college',
 424: 'goal',
 941: 'sunday',
 361: 'february',
 6: '2012',
 262: 'dance',
 8: '22nd',
 560: 'latino',
 604: 'march',
 2: '17',
 1013: 'university',
 848: 'saturday',
 1008: 'tutors',
 744: 'planet',
 485: 'human',
 602: 'mapping',
 420: 'give',
 1050: 'week',
 186: 'child',
 569: 'learn',
 796: 'read',
 926: 'storytelling',
 243: 'costume',
 597: 'making',
 912: 'stage',
 277: 'design',
 319: 'emergency',
 351: 'fair',
 17: '9th',
 1053: 'west',
 887: 'side',
 248: 'county',
 676: 'nutrition',
 314: 'educator',
 879: 'shape',
 306: 'east',
 13: '54st',
 801: 'rec',
 1046: 'water',
 45: 'aerobics',
 83: 'asser',
 573: 'levy',
 712: 'paint',
 57: 'alongside',
 783: 'publicolor',
 936: 'students',
 536: 'jumpstart',
 797: 'readers',
 564: 'lead',
 252: 'crafts',
 408: 'games',
 348: 'face',
 751: 'popcorn',
 527: 'jackie',
 835: 'robinson',
 716: 'parent',
 375: 'fitness',
 916: 'starrett',
 197: 'city',
 585: 'line',
 263: 'dancer',
 615: 'math',
 587: 'literacy',
 114: 'be',
 209: 'climb',
 985: 'top',
 608: 'marketing',
 86: 'assistant',
 313: 'education',
 673: 'nonprofit',
 867: 'seeks',
 805: 'recruitment',
 626: 'mentors',
 810: 'register',
 92: 'attend',
 142: 'breakfast',
 701: 'orientation',
 529: 'january',
 272: 'deliver',
 1058: 'winter',
 1031: 'visit',
 65: 'an',
 525: 'isolated',
 342: 'exercise',
 213: 'coach',
 670: 'night',
 115: 'beach',
 180: 'change',
 77: 'art',
 772: 'programs',
 229: 'consumer',
 779: 'protection',
 562: 'law',
 589: 'liver',
 579: 'life',
 565: 'leader',
 901: 'soup',
 547: 'kitchen',
 307: 'eastern',
 534: 'john',
 650: 'muir',
 930: 'street',
 1024: 'vendor',
 641: 'monthly',
 959: 'team',
 367: 'fiesta',
 977: 'throgs',
 658: 'neck',
 224: 'computer',
 956: 'teacher',
 567: 'leadership',
 244: 'council',
 693: 'opportunity',
 231: 'conversation',
 461: 'helpers',
 427: 'grades',
 714: 'pantry',
 288: 'distribution',
 305: 'earth',
 960: 'tech',
 1049: 'website',
 692: 'opportunities',
 175: 'cents',
 19: 'ability',
 203: 'classroom',
 877: 'set',
 146: 'brush',
 545: 'kindness',
 999: 'transportation',
 58: 'alternatives',
 129: 'bike',
 1020: 'valet',
 1026: 'video',
 311: 'editing',
 767: 'professionals',
 921: 'stipend',
 49: 'after',
 851: 'school',
 624: 'mentor',
 666: 'networking',
 138: 'bowling',
 398: 'fun',
 449: 'harlem',
 555: 'lanes',
 866: 'seeking',
 1078: 'yoga',
 902: 'spanish',
 695: 'or',
 389: 'french',
 362: 'feed',
 488: 'hungry',
 1080: 'yorkers',
 14: '55',
 690: 'only',
 735: 'phone',
 106: 'bank',
 819: 'representative',
 795: 'reach',
 704: 'out',
 643: 'morris',
 458: 'heights',
 904: 'special',
 155: 'camp',
 946: 'susan',
 551: 'komen',
 259: 'cure',
 433: 'greater',
 47: 'affiliate',
 303: 'dumbo',
 79: 'arts',
 698: 'organizational',
 148: 'budget',
 639: 'money',
 596: 'makes',
 871: 'sense',
 994: 'training',
 889: 'site',
 1027: 'videographer',
 376: 'fly',
 152: 'by',
 970: 'theater',
 429: 'grant',
 1074: 'writer',
 745: 'planning',
 778: 'proposal',
 759: 'preparation',
 399: 'fund',
 793: 'raising',
 450: 'harm',
 808: 'reduction',
 35: 'adv',
 515: 'intern',
 875: 'serving',
 575: 'lgbt',
 34: 'adults',
 482: 'how',
 830: 'ride',
 130: 'bikes',
 821: 'research',
 401: 'fundraising',
 280: 'developement',
 233: 'cook',
 840: 'row',
 50: 'afterschool',
 630: 'middle',
 885: 'shower',
 400: 'fundraisers',
 526: 'it',
 519: 'interpreters',
 563: 'lawyers',
 446: 'haitian',
 18: 'abe',
 757: 'pre',
 412: 'ged',
 640: 'monitor',
 89: 'astoria',
 634: 'million',
 1001: 'trees',
 421: 'giveaway',
 290: 'do',
 1081: 'you',
 1044: 'want',
 595: 'make',
 283: 'difference',
 204: 'classwish',
 896: 'snow',
 883: 'shoveling',
 196: 'citizenship',
 761: 'press',
 586: 'list',
 781: 'public',
 813: 'relations',
 743: 'plan',
 829: 'review',
 394: 'friendship',
 753: 'positive',
 121: 'beginnings',
 546: 'kit',
 611: 'mary',
 803: 'recreation',
 291: 'does',
 697: 'organization',
 659: 'need',
 858: 'search',
 928: 'strategy',
 332: 'esl',
 46: 'affected',
 924: 'storm',
 995: 'transform',
 590: 'lives',
 933: 'strengthen',
 220: 'communities',
 119: 'become',
 302: 'driver',
 1025: 'veterans',
 191: 'chinese',
 997: 'translator',
 512: 'instructors',
 653: 'museum',
 621: 'membership',
 275: 'department',
 284: 'director',
 117: 'beautify',
 996: 'transitional',
 822: 'residence',
 470: 'homeless',
 623: 'men',
 953: 'tank',
 517: 'internship',
 774: 'projects',
 841: 'run',
 1056: 'wild',
 139: 'boys',
 475: 'hope',
 419: 'girls',
 219: 'communications',
 792: 'raise',
 100: 'awareness',
 31: 'administrative',
 56: 'alliance',
 811: 'registrar',
 647: 'ms',
 1062: 'word',
 162: 'career',
 246: 'counselor',
 722: 'passover',
 304: 'early',
 188: 'childhood',
 149: 'build',
 747: 'plastic',
 137: 'bottle',
 857: 'sculpture',
 763: 'pride',
 523: 'is',
 538: 'just',
 76: 'around',
 238: 'corner',
 520: 'involved',
 675: 'now',
 390: 'fresh',
 53: 'air',
 957: 'teachers',
 372: 'find',
 729: 'perfect',
 533: 'job',
 684: 'office',
 1075: 'writing',
 264: 'data',
 326: 'entry',
 29: 'activism',
 738: 'photography',
 843: 'salesforce',
 265: 'database',
 261: 'customization',
 736: 'photo',
 333: 'essay',
 572: 'legal',
 42: 'advisor',
 467: 'hike',
 974: 'thon',
 236: 'coordinator',
 558: 'laser',
 950: 'tag',
 298: 'dowling',
 3: '175th',
 505: 'information',
 962: 'technology',
 352: 'fall',
 382: 'forest',
 826: 'restoration',
 541: 'kickoff',
 1002: 'trevor',
 582: 'lifeline',
 247: 'counselors',
 973: 'thomas',
 532: 'jefferson',
 614: 'materials',
 1076: 'year',
 386: 'founder',
 341: 'executive',
 453: 'haunted',
 557: 'lantern',
 989: 'tours',
 383: 'fort',
 986: 'totten',
 657: 'national',
 878: 'sexual',
 82: 'assault',
 689: 'online',
 993: 'trainers',
 48: 'african',
 63: 'american',
 210: 'clothing',
 301: 'drive',
 828: 'returning',
 865: 'seeds',
 939: 'success',
 746: 'plant',
 981: 'today',
 443: 'growth',
 1009: 'udec',
 328: 'enviromedia',
 636: 'mobile',
 606: 'maritime',
 102: 'bacchanal',
 742: 'pirates',
 365: 'fest',
 492: 'ikea',
 329: 'erie',
 111: 'basin',
 282: 'diabetes',
 88: 'association',
 364: 'feria',
 267: 'de',
 844: 'salud',
 664: 'nepali',
 105: 'bangla',
 784: 'punjabi',
 998: 'translators',
 674: 'not',
 769: 'profit',
 741: 'pioneer',
 159: 'capoeira',
 1023: 'various',
 752: 'positions',
 287: 'dispatcher',
 991: 'trainee',
 506: 'ing',
 603: 'marathon',
 388: 'free',
 593: 'love',
 135: 'books',
 268: 'dear',
 96: 'authors',
 52: 'aide',
 850: 'scheuer',
 627: 'merchandise',
 293: 'donate',
 943: 'supplies',
 360: 'feast',
 406: 'gala',
 112: 'battery',
 833: 'rise',
 919: 'stay',
 787: 'put',
 820: 'rescue',
 897: 'soccer',
 402: 'futsal',
 730: 'performing',
 36: 'advanced',
 202: 'classes',
 1070: 'world',
 854: 'science',
 1054: 'western',
 64: 'americorps',
 25: 'aces',
 310: 'economic',
 864: 'security',
 507: 'initiative',
 331: 'esi',
 633: 'mill',
 173: 'centers',
 631: 'midtown',
 1088: 'zumba',
 1030: 'vision',
 635: 'mission',
 66: 'analysis',
 552: 'lab',
 958: 'teaching',
 84: 'assist',
 827: 'resume',
 150: 'building',
 899: 'society',
 214: 'coaches',
 1040: 'vs',
 218: 'committee',
 842: 'russian',
 385: 'foster',
 170: 'celebration',
 616: 'may',
 7: '21th',
 688: 'one',
 711: 'pager',
 294: 'donation',
 489: 'hurricane',
 521: 'irene',
 354: 'far',
 836: 'rockaway',
 325: 'enjoy',
 1066: 'working',
 686: 'olympics',
 988: 'tournament',
 798: 'reading',
 719: 'partners',
 234: 'cooper',
 909: 'square',
 975: 'thrift',
 908: 'spring',
 166: 'case',
 599: 'management',
 404: 'fvcp',
 990: 'trail',
 254: 'crew',
 447: 'halloween',
 165: 'carnival',
 1042: 'walkathon',
 359: 'feasibility',
 67: 'analyst',
 749: 'police',
 868: 'seminar',
 1064: 'work',
 1035: 'visually',
 496: 'impaired',
 964: 'teens',
 972: 'this',
 322: 'energy',
 315: 'efficiency',
 321: 'end',
 859: 'season',
 156: 'campaign',
 123: 'benefits',
 802: 'reception',
 300: 'drill',
 237: 'copywriting',
 235: 'coord',
 454: 'have',
 725: 'penchant',
 55: 'all',
 971: 'things',
 1028: 'vintage',
 976: 'thriftshop',
 718: 'partner',
 726: 'pencil',
 720: 'partnership',
 710: 'packing',
 16: '8th',
 907: 'sports',
 346: 'expo',
 164: 'cares',
 184: 'cheerleaders',
 1045: 'wanted',
 445: 'habitat',
 371: 'finance',
 215: 'coffee',
 324: 'english',
 755: 'practice',
 570: 'learners',
 456: 'healthy',
 28: 'active',
 978: 'time',
 122: 'benefit',
 73: 'april',
 357: 'fashion',
 929: 'strawberry',
 87: 'assistants',
 174: 'central',
 1087: 'zoo',
 1: '125th',
 127: 'bideawee',
 440: 'greeters',
 592: 'looking',
 799: 'real',
 495: 'impact',
 504: 'inform',
 728: 'people',
 756: 'practices',
 580: 'lifebeat',
 413: 'general',
 932: 'streetsquash',
 286: 'discovery',
 874: 'services',
 663: 'neighborhood',
 768: 'profiles',
 951: 'take',
 915: 'stand',
 51: 'against',
 1029: 'violence',
 345: 'expert',
 41: 'advice',
 537: 'june',
 849: 'schedule',
 258: 'crowdfunding',
 727: 'penny',
 451: 'harvest',
 434: 'green',
 185: 'chefs',
 677: 'nutritionists',
 379: 'foodies',
 625: 'mentoring',
 136: 'boom',
 669: 'newsletter',
 217: 'come',
 934: 'strides',
 1043: 'walks',
 187: 'childcare',
 898: 'social',
 619: 'media',
 422: 'giving',
 157: 'can',
 61: 'ambassador',
 10: '2nd',
 967: 'thanksgiving',
 363: 'feeding',
 662: 'needy',
 782: 'publicity',
 723: 'patient',
 163: 'caregiver',
 1032: 'visiting',
 469: 'homebound',
 358: 'fc',
 679: 'nyawc',
 384: 'forum',
 21: 'about',
 1038: 'volunteering',
 809: 'refreshments',
 847: 'sara',
 837: 'roosevelt',
 206: 'cleanup',
 116: 'beautification',
 337: 'events',
 69: 'animal',
 484: 'hudson',
 834: 'river',
 605: 'mariners',
 825: 'response',
 343: 'exhibit',
 20: 'aboard',
 584: 'lilac',
 208: 'client',
 1052: 'welcome',
 279: 'desk',
 685: 'older',
 574: 'lexington',
 251: 'craft',
 750: 'poll',
 1065: 'workers',
 518: 'interperters',
 24: 'accounting',
 85: 'assistance',
 477: 'hosting',
 776: 'promotion',
 1011: 'unicef',
 954: 'tap',
 814: 'release',
 270: 'dedication',
 771: 'programming',
 500: 'incarnation',
 295: 'donor',
 544: 'kieran',
 906: 'sponsorship',
 1069: 'workshops',
 118: 'because',
 338: 'every',
 276: 'deserves',
 179: 'chance',
 740: 'pin',
 273: 'delivered',
 886: 'shred',
 15: '5th',
 99: 'avenue',
 169: 'cdsc',
 917: 'starving',
 78: 'artist',
 884: 'show',
 948: 'system',
 396: 'front',
 880: 'share',
 553: 'lanch',
 935: 'student',
 463: 'hemophilia',
 577: 'liason',
 629: 'methodist',
 476: 'hospital',
 113: 'bay',
 831: 'ridge',
 124: 'benonhurst',
 75: 'area',
 900: 'sought',
 97: 'autistic',
 297: 'douglaston',
 788: 'qns',
 812: 'registration',
 32: 'administrator',
 153: 'call',
 426: 'governor',
 804: 'recruiter',
 786: 'purim',
 327: 'envelope',
 938: 'stuffing',
 528: 'jam',
 462: 'helpline',
 923: 'store',
 374: 'first',
 415: 'generation',
 1022: 'van',
 241: 'cortlandt',
 816: 'remembrance',
 945: 'survey',
 823: 'resonations',
 143: 'breast',
 323: 'engine',
 694: 'optimization',
 622: 'memorial',
 894: 'sloan',
 540: 'kettering',
 435: 'greenhouse',
 436: 'greening',
 227: 'concert',
 334: 'evacuation',
 824: 'resources',
 417: 'gift',
 126: 'bicycling',
 656: 'my',
 393: 'friends',
 473: 'honor',
 1051: 'weekend',
 731: 'person',
 651: 'mural',
 312: 'editor',
 732: 'personal',
 882: 'shopper',
 764: 'pro',
 134: 'bono',
 253: 'create',
 160: 'cards',
 920: 'step',
 672: 'non',
 780: 'provider',
 516: 'interns',
 645: 'motion',
 431: 'graphics',
 125: 'best',
 147: 'buddies',
 502: 'inern',
 103: 'back',
 588: 'little',
 242: 'cosmetologist',
 107: 'barber',
 1036: 'vocational',
 72: 'apartment',
 439: 'greeter',
 766: 'professional',
 1019: 'use',
 893: 'skills',
 702: 'others',
 369: 'figure',
 257: 'croton',
 190: 'chinatown',
 193: 'ci',
 758: 'prep',
 239: 'corporate',
 1063: 'wordpress',
 132: 'blog',
 510: 'instructer',
 807: 'red',
 474: 'hook',
 289: 'divert',
 966: 'textiles',
 395: 'from',
 554: 'landfill',
 437: 'greenmarket',
 965: 'textile',
 154: 'calling',
 195: 'citizens',
 497: 'improve',
 26: 'achievement',
 721: 'passion',
 481: 'housing',
 1067: 'works',
 499: 'inc',
 441: 'group',
 299: 'drama',
 561: 'laundromats',
 320: 'employment',
 927: 'strategic',
 667: 'never',
 104: 'bad',
 391: 'friend',
 403: 'future',
 201: 'class',
 1059: 'wish',
 387: 'fpcj',
 1072: 'worship',
 1010: 'undergraduate',
 428: 'graduate',
 228: 'conference',
 1047: 'we',
 775: 'promote',
 550: 'knowledge',
 715: 'parade',
 74: 'archivist',
 425: 'google',
 44: 'adwords',
 493: 'imentor',
 642: 'more',
 598: 'male',
 632: 'miles',
 637: 'moms',
 183: 'charity',
 176: 'century',
 987: 'tour',
 198: 'civil',
 724: 'patrol',
 62: 'america',
 539: 'kept',
 862: 'secret',
 648: 'ms131',
 549: 'knitter',
 256: 'crochet',
 131: 'blankets',
 177: 'ceo',
 591: 'logo',
 1012: 'unique',
 1057: 'will',
 128: 'big',
 37: 'adventure',
 23: 'accountant',
 876: 'session',
 888: 'single',
 644: 'mothers',
 192: 'choice',
 895: 'smc',
 1055: 'wii',
 705: 'outdoor',
 671: 'nights',
 607: 'market',
 514: 'intake',
 638: 'monday',
 141: 'branding',
 140: 'brand',
 491: 'identity',
 649: 'mt',
 1086: 'zion',
 543: 'kidz',
 817: 'reorganize',
 578: 'library',
 378: 'food',
 91: 'athletic',
 568: 'league',
 655: 'musician',
 59: 'alzheimer',
 654: 'music',
 109: 'bash',
 765: 'proctor',
 952: 'taking',
 339: 'exams',
 777: 'promotional'}

In [38]:
def return_weights(vocab, original_vocab, vector, vector_index, top_n):
    # Create a dictionary where:
    # key = feature (word) index
    # value = TF-IDF weight for the selected document
    zipped = dict(zip(vector[vector_index].indices,
                      vector[vector_index].data))

    # Convert the dictionary to a pandas Series
    # This makes it easy to sort the TF-IDF weights
    zipped_series = pd.Series(zipped)

    # Sort the TF-IDF weights in descending order,
    # select the top_n highest-weighted features,
    # and return their feature indices as a list
    return zipped_series.sort_values(ascending=False).head(top_n).index.tolist()

In [39]:
def words_to_filter(vocab, original_vocab, vector, top_n):
    # Create an empty list to store the top feature indices
    # from every document
    filter_list = []

    # Loop through each document (row) in the TF-IDF matrix
    for i in range(vector.shape[0]):

        # Get the indices of the top_n highest-weighted
        # features for the current document
        filtered = return_weights(vocab, original_vocab, vector, i, top_n)

        # Add these feature indices to the filter list
        filter_list.extend(filtered)

    # Convert the list to a set to remove duplicate indices
    return set(filter_list)


# Get the unique feature indices that are among the top 3
# weighted words across all documents
filtered_words = words_to_filter(
    vocab,
    tfidf_vec.vocabulary_,
    text_tfidf,
    3
)

# Keep only the columns (features) whose indices are in
# filtered_words, creating a reduced TF-IDF matrix
filtered_text = text_tfidf[:, list(filtered_words)]

In [40]:
# Split the dataset according to the class distribution of category_desc
X_train, X_test, y_train, y_test = train_test_split(filtered_text.toarray(), y, stratify=y, random_state=42)

# Fit the model to the training data
nb.fit(X_train, y_train)

# Print out the model's accuracy
print(nb.score(X_test, y_test))

0.5989304812834224


In [41]:
wine_df.columns

Index(['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium',
       'total_phenols', 'flavanoids', 'nonflavanoid_phenols',
       'proanthocyanins', 'color_intensity', 'hue',
       'od280/od315_of_diluted_wines', 'proline', 'Proline_log'],
      dtype='object')

In [42]:
wine_df['type'] = pd.qcut(wine_df.index, q=3, labels=[1, 2, 3])

In [56]:
churn_df['Partner'].unique()

array(['Yes', 'No'], dtype=object)

In [44]:
# Instantiate a PCA object
pca = PCA()

# Define the features and labels from the wine dataset
X = wine_df.drop("type", axis =1)
y = wine_df["type"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Apply PCA to the wine dataset X vector
pca_X_train = pca.fit_transform(X_train)
pca_X_test = pca.transform(X_test)

# Look at the percentage of variance explained by the different components
print(pca.explained_variance_ratio_)

[9.97731187e-01 2.07181491e-03 1.09749577e-04 5.74081356e-05
 1.29890746e-05 9.32545300e-06 3.12896844e-06 1.65842608e-06
 1.15915810e-06 8.44409273e-07 3.98628843e-07 2.07914715e-07
 8.55502334e-08 4.28964306e-08]


In [45]:
# Fit knn to the training data
knn.fit(pca_X_train, y_train)

# Score knn on the test data and print it out
print(knn.score(pca_X_test, y_test))

0.7111111111111111


In [50]:
wine_df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,Proline_log,type
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,6.970730,1
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,6.956545,1
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,7.077498,1
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,7.299797,1
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,6.599870,1


In [57]:
# Use pandas to encode us values as 1 and others as 0
churn_df["Partner_enc"] = churn_df["Partner"].apply(lambda value : 1 if value == "us" else 0)

# Print the number of unique type values
print(len(churn_df['Partner'].unique()))

# Create a one-hot encoded set of the type values
type_set = pd.get_dummies(churn_df['Partner'])

# Concatenate this set back to the ufo DataFrame
churn_df = pd.concat([churn_df, type_set], axis=1)

2
